In [5]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import time
from datetime import datetime, timedelta
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/campechehoy/articles.parquet"

# Base URL format for daily pages
base_url = "http://campechehoy.mx/{year}/{month:02d}/{day:02d}/"

def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title.
    """
    pattern = r'\d{4}/\d{2}/\d{2}/[a-zA-Z0-9-]+'
    return re.search(pattern, url)

def extract_article_data(url, y, m, d):
    """
    Extract and return the title, main text, date, and source from an article page.
    """
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the title
        title_tag = soup.find('h1', class_='zox-post-title left entry-title')
        title = title_tag.get_text(strip=True) if title_tag else 'No title found'

        # Extract the main text
        dd_content = soup.find_all('div', class_='zox-post-main')
        main_text = ' '.join(p.get_text(strip=True) for div in dd_content for p in div.find_all('p'))

        try:
            subtitle_tag = soup.find('span', class_='zox-post-excerpt')
            subtitle = ' '.join(p.get_text(strip=True) for div in subtitle_tag for p in div.find_all('p'))
        except: 
            subtitle = None

        # Extract the source
        category_tag = soup.find('span', class_='zox-post-cat')
        category = category_tag.get_text(strip=True) if title_tag else 'No category found'

        return {
            'url': url,
            'title': title,
            'sub_title':subtitle,
            'main_text': main_text.strip(),
            'date': f'{y}-{m:02d}-{d:02d}',
            'topic': category
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return None
    
def crawl_daily_page(year, month, day):
    """
    Crawl a daily page, find all subpages, and extract article links while removing non-article URLs.
    """
    url = base_url.format(year=year, month=month, day=day)
    daily_data = []  # Store daily data temporarily for this day
    
    try:
        # Extract subpage links from the main daily page
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        print(f"Accessing main page: {url}")
            
        unique_links = set()
        for link in soup.find_all("a", href=True):
            href = urljoin(url, link['href'])
            if is_relevant_url(href):
                unique_links.add(href)  # Add only unique links

        # Extract and store article data from unique pages
        for href in unique_links:
            article = extract_article_data(href, y = year, m = month, 
                                           d = day)
            if article:
                daily_data.append(article)
                print(f"Extracted data from {href}")


    except requests.exceptions.RequestException as e:
        print(f"Error accessing daily page {url}: {e}")

    return daily_data

# Function to save data to parquet
def save_to_parquet(data, parquet_file):
    df = pd.DataFrame(data)
    if not df.empty:
        if os.path.exists(parquet_file):
            initial = pd.read_parquet(parquet_file)
            df = pd.concat([initial, df]).reset_index(drop=True)
            df = df.drop_duplicates('url').reset_index(drop = True)
            df.to_parquet(parquet_file, index=False,
                           engine="pyarrow", compression="gzip")
        else:
            df.to_parquet(parquet_file, index = False,
                           engine="pyarrow", compression="gzip")

# Determine the start date by checking the existing Parquet file
if os.path.exists(parquet_file):
    existing_data = pd.read_parquet(parquet_file)
    last_date_str = existing_data['date'].max()
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    start_date = last_date + timedelta(days=1)
    print(f"Resuming crawl from {start_date.date()}")
else:
    start_date = datetime(2016, 1, 1)

# Set the end date to January 31, 2024
end_date = datetime(2025, 1, 15)

# Crawl and save articles day-by-day, resuming if interrupted
current_date = start_date
while current_date <= end_date:
    daily_data = crawl_daily_page(current_date.year, current_date.month, current_date.day)
    if daily_data:
        save_to_parquet(daily_data, parquet_file)
    current_date += timedelta(days=1)
    print(f"Completed crawling for {current_date.date() - timedelta(days=1)}")

print("Crawling completed or paused; data saved in Parquet format.")


Error accessing daily page http://campechehoy.mx/2016/01/01/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/01/
Completed crawling for 2016-01-01
Error accessing daily page http://campechehoy.mx/2016/01/02/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/02/
Completed crawling for 2016-01-02
Error accessing daily page http://campechehoy.mx/2016/01/03/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/03/
Completed crawling for 2016-01-03
Error accessing daily page http://campechehoy.mx/2016/01/04/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/04/
Completed crawling for 2016-01-04
Error accessing daily page http://campechehoy.mx/2016/01/05/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/05/
Completed crawling for 2016-01-05
Error accessing daily page http://campechehoy.mx/2016/01/06/: 404 Client Error: Not Found for url: http://campechehoy.mx/2016/01/06/
Completed crawling for 2016-01-0